In [ ]:
import copy
import json
import re
from pathlib import Path

root = Path.cwd()
file_paths = [
    root / 'server' / 'data' / 'few-shot-scenarios.json',
    root / 'server' / 'few-shot-scenarios.json',
]

def load_json(path: Path):
    text = path.read_text(encoding='utf-8')
    if not text.strip() and path.name == 'few-shot-scenarios.json' and path.parent.name == 'server':
        return None, text
    return json.loads(text), text

def parse_systolic(bp):
    if isinstance(bp, (int, float)):
        return int(bp)
    if isinstance(bp, str):
        match = re.search(r'(\d{2,3})\s*/', bp)
        if match:
            return int(match.group(1))
        match = re.search(r'-?\d+(?:\.\d+)?', bp)
        if match:
            return int(float(match.group(0)))
    return None

def hr_rhythm(ecg):
    mapping = {
        'Atrial Fibrillation': 'irregular',
        'Atrial Flutter': 'regular',
        'SVT': 'regular',
        'Ventricular Tachycardia': 'regular',
        'Sinus Bradycardia': 'regular',
        'Sinus Tachycardia': 'regular',
        'Normal Sinus Rhythm': 'regular',
    }
    return mapping.get(str(ecg or '').strip(), 'regular')

def hr_volume(bp):
    systolic = parse_systolic(bp)
    if systolic is not None and systolic < 100:
        return 'weak'
    return 'strong'

def numeric_token(value):
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if int(value) == value:
            return str(int(value))
        return str(value)
    if isinstance(value, str):
        match = re.search(r'-?\d+(?:\.\d+)?', value.strip())
        if match:
            token = match.group(0)
            return str(int(float(token))) if float(token).is_integer() else token
    return None

def normalize_hr(vital_set):
    raw = vital_set.get('hr')
    if isinstance(raw, str) and raw.strip().upper() == 'N/A':
        return 'N/A'
    if isinstance(raw, str) and ',' in raw:
        return raw
    rate = numeric_token(raw) or str(raw).strip()
    if not rate:
        return raw
    return f"{rate}, {hr_rhythm(vital_set.get('ecgInterpretation'))}, {hr_volume(vital_set.get('bp'))}"

def normalize_rr(vital_set):
    raw = vital_set.get('rr')
    if isinstance(raw, str) and ',' in raw:
        return raw
    raw_text = str(raw).strip()
    if not raw_text:
        return raw
    if re.search(r'Assisted|BVM', raw_text, re.IGNORECASE):
        return f"{raw_text}, regular, assisted"
    token = numeric_token(raw)
    if token is not None:
        rate_value = float(token)
        volume = 'shallow' if rate_value >= 30 or rate_value <= 10 else 'full'
        return f"{token}, regular, {volume}"
    return f"{raw_text}, regular, full"

def transform_vital_set(vital_set):
    if not isinstance(vital_set, dict):
        return 0
    before = copy.deepcopy(vital_set)
    for key in ('hrRhythm', 'hrVolume', 'rrRhythm', 'rrVolume'):
        vital_set.pop(key, None)
    vital_set['hr'] = normalize_hr(vital_set)
    vital_set['rr'] = normalize_rr(vital_set)
    return 1 if vital_set != before else 0

def transform_scenarios(scenarios):
    changed_sets = 0
    for scenario in scenarios:
        vital_signs = scenario.get('vitalSigns')
        if not isinstance(vital_signs, dict):
            continue
        changed_sets += transform_vital_set(vital_signs.get('firstSet'))
        changed_sets += transform_vital_set(vital_signs.get('secondSet'))
        additional_sets = vital_signs.get('additionalSets')
        if isinstance(additional_sets, list):
            for vital_set in additional_sets:
                changed_sets += transform_vital_set(vital_set)
    return changed_sets

primary_data, primary_text = load_json(file_paths[0])
secondary_data, secondary_text = load_json(file_paths[1])
if secondary_data is None:
    secondary_data = copy.deepcopy(primary_data)

primary_changed = transform_scenarios(primary_data)
secondary_changed = transform_scenarios(secondary_data)

file_paths[0].write_text(json.dumps(primary_data, indent=2) + '\n', encoding='utf-8')
file_paths[1].write_text(json.dumps(secondary_data, indent=2) + '\n', encoding='utf-8')

def sample_snippet(scenarios):
    for scenario in scenarios:
        vital_signs = scenario.get('vitalSigns', {})
        for key in ('firstSet', 'secondSet'):
            vital_set = vital_signs.get(key)
            if isinstance(vital_set, dict):
                return {
                    'context': vital_set.get('context'),
                    'hr': vital_set.get('hr'),
                    'rr': vital_set.get('rr'),
                }
        additional_sets = vital_signs.get('additionalSets') or []
        for vital_set in additional_sets:
            if isinstance(vital_set, dict):
                return {
                    'context': vital_set.get('context'),
                    'hr': vital_set.get('hr'),
                    'rr': vital_set.get('rr'),
                }
    return {}

report = {
    'both_files_updated': True,
    'files': [
        {
            'path': str(file_paths[0].relative_to(root)),
            'changed_vital_sets': primary_changed,
            'sample': sample_snippet(primary_data),
        },
        {
            'path': str(file_paths[1].relative_to(root)),
            'changed_vital_sets': secondary_changed,
            'sample': sample_snippet(secondary_data),
            'used_primary_fallback': not secondary_text.strip(),
        },
    ],
}

print(json.dumps(report, indent=2))